# **Face Detection using MTCNN**

**Goal**: To use a pre-trained Multi-task Cascaded Convolutional Neural Network (MTCNN) for face detection. Loading the model from facenet_pytorch library and using it on images extracted from Mary Kom YouTube interviews video. 

**Objectives**:

- Initialize a pre-trained MTCNN model from facenet_pytorch.

- Detect faces in an image using MTCNN model.

- Display the resulting bounding boxes of faces detected by the model.

- Crop out detected faces for futher analysis.

- Determine facial landmarks such as eyes, nose, and mouth using the MTCNN model.

- Select a subset of images for face recognition taks. 

In [14]:
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt 
import PIL
import torch
import torchvision
from facenet_pytorch import MTCNN
from PIL import Image
from torchvision.utils import make_grid

In [15]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)
print("PIL version : ", PIL.__version__)

Platform: win32
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
---
torch version :  2.5.1+cpu
torchvision version :  0.20.1+cpu
PIL version :  10.2.0


In [16]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using {device} device.")

Using cpu device.


**Initialize MTCNN Model**

keep all detected faces and set the minimum size to search for to be 60


    - device: The device on which to run the model.

    - keep_all: A boolean determining if all detected faces are returned or not.

    - min_face_size: Minimum face size (in pixels) to search for in the image.
    
    - post_process: A boolean determining if we want image standardization of detected faces. This is advised before proceeding with face recognition models, but if we want face images that are returned to us to look normal to the human eye, we can set post_process=False.


In [17]:
mtcnn = MTCNN(device=device, keep_all=True, min_face_size=60, post_process=False)

print(mtcnn)

MTCNN(
  (pnet): PNet(
    (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=10)
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=16)
    (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (prelu3): PReLU(num_parameters=32)
    (conv4_1): Conv2d(32, 2, kernel_size=(1, 1), stride=(1, 1))
    (softmax4_1): Softmax(dim=1)
    (conv4_2): Conv2d(32, 4, kernel_size=(1, 1), stride=(1, 1))
  )
  (rnet): RNet(
    (conv1): Conv2d(3, 28, kernel_size=(3, 3), stride=(1, 1))
    (prelu1): PReLU(num_parameters=28)
    (pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv2): Conv2d(28, 48, kernel_size=(3, 3), stride=(1, 1))
    (prelu2): PReLU(num_parameters=48)
    (pool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
    (conv3): Conv2d(48, 64,

c:\Users\Asus\AppData\Local\Programs\Python\Python312\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.

**Create a variable for the current working directory using pathlib syntax.**

In [18]:
curr_work_dir = Path.cwd()

print(curr_work_dir)

d:\workspace\applied-ai\0x03-celebrity_sightings_in_india


**Create an absolute path for the extracted_frames directory using the pathlib syntax.**

In [19]:
extracted_frames_dir = curr_work_dir/"data"/"extracted_frames"

print(extracted_frames_dir)

d:\workspace\applied-ai\0x03-celebrity_sightings_in_india\data\extracted_frames


**File path to the sample images that we'll use**